# Qwen3.6-35B-A3B → Cloudflare Tunnel → OpenCode  (200K context, large output)
**Kaggle settings:** Accelerator **GPU T4 x2**, Internet **On**. Run the 4 cells in order.
Downloads a prebuilt CUDA `llama-server` (no 20-minute compile) while the model downloads in parallel. Compiles from source only if no prebuilt binary works.

**Long-context notes**
- Qwen3.6-35B-A3B is a hybrid model (mostly linear-attention layers), so its KV cache is small: 200K tokens ≈ 2–3 GB at q8_0. VRAM is mainly the ~22 GB of weights, so 2xT4 (~30 GB) is tight but workable. Cell 3 automatically falls back to a q4_0 KV cache, then a smaller context, if loading runs out of memory.
- The first 200K-token prefill on a T4 can take many minutes. Later turns reuse the prompt cache, so they are fast. Cloudflare quick tunnels drop requests that send no bytes for ~100 s, so avoid dumping a huge fresh prompt in one go; build context up gradually.

In [ ]:
# 1 ── Config
import os, secrets
QUANT   = "UD-Q4_K_XL"        # ~22 GB. If 200K won't fit, use "UD-Q3_K_XL" (~16 GB, more headroom)
CTX     = 204800              # 200K context window (model native max is 262144)
OUT     = 65536               # max output tokens advertised to opencode (thinking + answer)
KV_TYPE = "q8_0"              # KV cache precision; cell 3 falls back to q4_0 on OOM
UBATCH  = 1024                # prefill micro-batch: bigger = faster prompt processing, more VRAM (falls back to 512)
BIN_REPO = ""                 # optional private HF repo to cache the compiled llama-server, e.g. "yourname/llama-server-t4"
                              # (needs an HF_TOKEN with write access, as env var or Kaggle secret). First run compiles
                              # and uploads; every later run downloads it in seconds.
ALIAS   = "qwen3.6-35b-a3b"   # model id in opencode
API_KEY = os.environ.get("LLM_API_KEY") or "sk-" + secrets.token_hex(16)
print("API key:", API_KEY)

In [ ]:
# 2 ── Install: llama-server (cached / prebuilt / compiled) + model, in parallel
import os, re, json, glob, time, tarfile, zipfile, shutil, threading, subprocess, urllib.request
W = "/kaggle/working"; os.makedirs(W, exist_ok=True)
os.environ["PATH"] = "/usr/local/cuda/bin:" + os.environ["PATH"]
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"   # replaces the deprecated hf_transfer switch
M = "/tmp/models"; os.makedirs(M, exist_ok=True)   # /kaggle/working is capped at ~20 GB; /tmp is not
os.environ["HF_HOME"] = "/tmp/hf"

def sh(cmd):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True, executable="/bin/bash")
    return p.returncode, p.stdout + p.stderr

print(sh("nvidia-smi -L")[1] or "!! NO GPU - enable GPU T4 x2")
sh("pip -q install -U huggingface_hub hf_xet ninja")
from huggingface_hub import snapshot_download

# ---- model download (background thread)
res = {}
def dl_model():
    try:
        snapshot_download("unsloth/Qwen3.6-35B-A3B-GGUF", allow_patterns=[f"*{QUANT}*.gguf"], local_dir=M)
        res["ok"] = True
    except Exception as e: res["err"] = e
t_model = threading.Thread(target=dl_model); t_model.start()

# ---- binary helpers
def find_bin(d):
    c = [p for p in glob.glob(f"{d}/**/llama-server", recursive=True) if os.path.isfile(p)]
    return c[0] if c else None

def hf_token():
    if os.environ.get("HF_TOKEN"): return os.environ["HF_TOKEN"]
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception: return None

def cuda_libs():
    """The prebuilt needs CUDA 12 runtime libs this Kaggle image may lack: get them from pip (~1 min), no compile."""
    sh("pip -q install nvidia-cuda-runtime-cu12 nvidia-cublas-cu12")
    import site
    dirs = []
    for sp in site.getsitepackages() + [site.getusersitepackages()]:
        dirs += [d for d in glob.glob(f"{sp}/nvidia/*/lib") if os.path.isdir(d)]
    return ":".join(dirs)

NV = None
def works(b):
    global NV
    os.chmod(b, 0o755)
    if NV is None: NV = cuda_libs()
    ld = f"{os.path.dirname(b)}:{NV}:/usr/local/cuda/lib64"
    env = f"LD_LIBRARY_PATH={ld}:$LD_LIBRARY_PATH"
    rc, out = sh(f"{env} ldd {b} | grep 'not found'")
    if out.strip(): print("missing libs:", out.strip()[:300]); return False
    rc, out = sh(f"{env} {b} --version")
    if rc != 0: print("does not run:", out[-300:]); return False
    print(out.strip()[-200:])
    os.environ["LD_LIBRARY_PATH"] = ld + ":" + os.environ.get("LD_LIBRARY_PATH", "")
    return True

# 1) a binary you attached as a Kaggle dataset (Add Input -> dataset containing `llama-server`)
def try_kaggle_input():
    for f in glob.glob("/kaggle/input/*/llama-server") + glob.glob("/kaggle/input/*/*/llama-server"):
        d = f"{W}/llama-bin-input"; os.makedirs(d, exist_ok=True); b = f"{d}/llama-server"
        shutil.copy(f, b)
        if works(b): print("using binary from Kaggle input:", f); return b

# 2) binary cached in your private HF repo
def try_hf_cache():
    if not BIN_REPO: return None
    try:
        from huggingface_hub import hf_hub_download
        b = hf_hub_download(BIN_REPO, "llama-server", token=hf_token(), local_dir=f"{W}/llama-bin-cached")
        if works(b): print("using cached binary from", BIN_REPO); return b
    except Exception as e: print("no cached binary yet:", repr(e)[:150])

# 3) upstream prebuilt CUDA release (upstream often ships no Linux CUDA build, in which case this falls through)
# CUDA 12 builds only (Kaggle has no CUDA 13 libs)
SOURCES = [("ggml-org/llama.cpp", lambda n: "ubuntu" in n and "cuda-12" in n and "x64" in n and "cudart" not in n),
           ("ai-dock/llama.cpp-cuda", lambda n: "cuda-12.8" in n and "amd64" in n)]
def try_prebuilt():
    for repo, ok in SOURCES:
        try:
            r = urllib.request.Request(f"https://api.github.com/repos/{repo}/releases/latest", headers={"User-Agent": "kaggle"})
            assets = json.load(urllib.request.urlopen(r, timeout=30))["assets"]
            a = next((x for x in assets if ok(x["name"]) and x["name"].endswith((".tar.gz", ".tgz", ".zip"))), None)
            if not a: print(repo, "no matching CUDA asset; has:", [x["name"] for x in assets][:12]); continue
            d = f"{W}/llama-bin"; os.makedirs(d, exist_ok=True); f = f"{d}/{a['name']}"
            urllib.request.urlretrieve(a["browser_download_url"], f)
            if f.endswith(".zip"): zipfile.ZipFile(f).extractall(d)
            else:
                with tarfile.open(f) as t:
                    try: t.extractall(d, filter="data")      # safe extraction (Python 3.12+)
                    except TypeError: t.extractall(d)       # older Python without the filter argument
            b = find_bin(d)
            if b and works(b): print("prebuilt OK:", repo, a["name"]); return b
        except Exception as e: print(repo, "failed:", repr(e)[:200])

# 4) compile (static single binary, T4-only, no tests/examples, Ninja)
def compile_from_source():
    print("compiling from source (T4 only, static, Ninja). Progress below.")
    libs = sh("ldconfig -p | grep 'libcuda\\.so'; find / -name 'libcuda.so*' -not -path '/proc/*' 2>/dev/null")[1]
    paths = [l.split()[-1] for l in libs.splitlines() if os.path.exists(l.split()[-1])] if libs.strip() else []
    os.makedirs(f"{W}/stubs", exist_ok=True); link = f"{W}/stubs/libcuda.so"
    if os.path.lexists(link): os.remove(link)
    if paths: os.symlink(os.path.realpath(paths[0]), link)
    else: sh(f'echo "" | gcc -shared -x c - -Wl,-soname,libcuda.so.1 -o {link}')
    sh(f"rm -rf {W}/llama.cpp && git clone --depth 1 https://github.com/ggml-org/llama.cpp {W}/llama.cpp")
    flags = "-Wl,--allow-shlib-undefined"
    rc, out = sh(f"cd {W}/llama.cpp && cmake -G Ninja -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 "
                 f"-DBUILD_SHARED_LIBS=OFF -DGGML_NATIVE=OFF -DGGML_CUDA_FA_ALL_QUANTS=OFF "
                 f"-DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release "
                 f"-DCUDA_cuda_driver_LIBRARY={link} -DCUDA_CUDA_LIBRARY={link} -DCMAKE_LIBRARY_PATH={W}/stubs "
                 f"-DCMAKE_EXE_LINKER_FLAGS={flags} -DCMAKE_SHARED_LINKER_FLAGS={flags}")
    assert rc == 0, out[-3000:]
    p = subprocess.Popen(f"cd {W}/llama.cpp && cmake --build build --target llama-server",
                         shell=True, executable="/bin/bash", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    last, tail = 0, []
    for line in p.stdout:
        tail = (tail + [line])[-40:]
        if re.match(r"\[\s*\d+[%/]", line) and time.time() - last > 15: print(line.strip()[:100], flush=True); last = time.time()
    p.wait(); assert p.returncode == 0, "".join(tail)
    b = f"{W}/llama.cpp/build/bin/llama-server"
    # keep a copy so you never compile again
    shutil.copy(b, f"{W}/llama-server")        # downloadable from the notebook's Output tab -> upload as a Kaggle dataset
    if BIN_REPO and hf_token():
        try:
            from huggingface_hub import HfApi
            api = HfApi(token=hf_token()); api.create_repo(BIN_REPO, private=True, exist_ok=True)
            api.upload_file(path_or_fileobj=b, path_in_repo="llama-server", repo_id=BIN_REPO)
            print("uploaded compiled binary to", BIN_REPO, "- next run will skip compiling")
        except Exception as e: print("HF upload failed:", repr(e)[:200])
    return b

BIN = (find_bin(f"{W}/llama-bin") or find_bin(f"{W}/llama.cpp/build") or
       try_kaggle_input() or try_hf_cache() or try_prebuilt() or compile_from_source())
print("BIN:", BIN)

t_model.join()
assert res.get("ok"), res.get("err")
MODEL = [f for f in sorted(glob.glob(f"{M}/**/*{QUANT}*.gguf", recursive=True)) if "mmproj" not in f][0]
print("MODEL:", MODEL)

In [ ]:
# 3 ── Start server + Cloudflare tunnel, test, print opencode.json
import subprocess, time, re, json, requests, os

def launch(kv, ctx, ub):
    cmd = [BIN, "-m", MODEL, "--alias", ALIAS, "--host", "127.0.0.1", "--port", "8080",
           "-ngl", "999", "--split-mode", "layer", "--tensor-split", "1,1", "--flash-attn", "on",
           "-c", str(ctx), "--cache-type-k", kv, "--cache-type-v", kv,
           "-b", "2048", "-ub", str(ub), "-t", str(os.cpu_count()), "--parallel", "1", "--jinja",
           "-n", "-1",                      # no server-side cap on output length
           "--temp", "0.6", "--top-p", "0.95", "--top-k", "20", "--min-p", "0.0", "--api-key", API_KEY]
    srv = subprocess.Popen(cmd, stdout=open(f"{W}/llama-server.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(240):
        try:
            if requests.get("http://127.0.0.1:8080/health", timeout=2).status_code == 200: return srv
        except Exception: pass
        if srv.poll() is not None: return None
        time.sleep(5)
    srv.kill(); return None

# try the preferred setup first, then progressively lighter ones if VRAM runs out
ATTEMPTS = [(KV_TYPE, CTX, UBATCH), (KV_TYPE, CTX, 512), ("q4_0", CTX, 512), ("q4_0", 131072, 512)]
server = None
for kv, ctx, ub in ATTEMPTS:
    print(f"starting llama-server: ctx={ctx}, KV={kv}, ubatch={ub} ...")
    server = launch(kv, ctx, ub)
    if server: CTX = ctx; print(f"server ready (ctx={ctx}, KV={kv}, ubatch={ub})"); break
    print("failed, log tail:\n", open(f"{W}/llama-server.log").read()[-1500:])
    subprocess.run("pkill -f llama-server", shell=True); time.sleep(5)
if not server: raise SystemExit("server could not start - try QUANT='UD-Q3_K_XL' in cell 1")
print(sh("nvidia-smi --query-gpu=index,memory.used,memory.total --format=csv")[1])

CF = f"{W}/cloudflared"
if not os.path.exists(CF):
    sh(f"curl -sL -o {CF} https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x {CF}")
tunnel = subprocess.Popen([CF, "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8080"],
                          stdout=open(f"{W}/cloudflared.log", "w"), stderr=subprocess.STDOUT)
URL = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(f"{W}/cloudflared.log").read())
    if m: URL = m.group(0); break
    time.sleep(1)
assert URL, open(f"{W}/cloudflared.log").read()[-2000:]

# local test (always works if the server is up)
hdr = {"Authorization": f"Bearer {API_KEY}"}
body = {"model": ALIAS, "messages": [{"role": "user", "content": "Say hi in 5 words."}], "max_tokens": 300}
r = requests.post("http://127.0.0.1:8080/v1/chat/completions", headers=hdr, json=body)
print("local test:", r.status_code, r.json()["choices"][0]["message"].get("content"))

# wait for the tunnel to register, then try the public URL (non-fatal: new hostnames can take ~1 min to resolve,
# and Kaggle's DNS may cache the first failure; what matters is that YOUR machine can reach it)
for _ in range(30):
    if "Registered tunnel connection" in open(f"{W}/cloudflared.log").read(): break
    time.sleep(2)
for i in range(8):
    try:
        r = requests.post(URL + "/v1/chat/completions", headers=hdr, json=body, timeout=60)
        print("public test:", r.status_code, r.json()["choices"][0]["message"].get("content")); break
    except Exception as e:
        print(f"public test try {i+1}: not reachable from Kaggle yet ({type(e).__name__})"); time.sleep(10)
else:
    print("Kaggle could not resolve the tunnel URL. This is usually fine - test it from your own machine with:")
    print(f'  curl {URL}/v1/models -H "Authorization: Bearer {API_KEY}"')

cfg = {"$schema": "https://opencode.ai/config.json",
       "provider": {"kaggle": {"npm": "@ai-sdk/openai-compatible", "name": "Kaggle Qwen3.6",
                    "options": {"baseURL": URL + "/v1", "apiKey": API_KEY,
                                "timeout": False},   # long prefills/generations: don't let opencode time out
                    "models": {ALIAS: {"name": "Qwen3.6 35B-A3B", "limit": {"context": CTX, "output": OUT}}}}},
       "model": f"kaggle/{ALIAS}"}
print("\nSave on YOUR machine as opencode.json:\n"); print(json.dumps(cfg, indent=2))

In [ ]:
# 4 ── Keep alive (leave running; interrupt to stop)
import time
try:
    while server.poll() is None and tunnel.poll() is None: time.sleep(30)
finally:
    print("server:", server.poll(), "tunnel:", tunnel.poll()); print(open(f"{W}/llama-server.log").read()[-2000:])